# Практика: membership, счётчики и сегменты

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")


## 1. Контрольный список

Проверьте ids через set, верните dict id -> bool.

In [ ]:
known = set(df["order_id"])
probes = df["order_id"].sample(min(8, len(df)), random_state=52).tolist() + ["missing"]
check = None  # TODO
assert isinstance(check, dict) and len(check) == len(probes)
assert check["missing"] is False


## 2. Пересечение множеств

Сравните customer_state и seller_state.

In [ ]:
customer_states = None  # TODO
seller_states = None    # TODO
shared_states = None    # TODO
assert shared_states == customer_states & seller_states
assert shared_states <= customer_states and shared_states <= seller_states


## 3. Счётчик пар регионов

Ключ `(seller_state, customer_state)`.

In [ ]:
pair_total = {}  # TODO
assert sum(pair_total.values()) == len(df)
assert all(isinstance(key, tuple) and len(key) == 2 for key in pair_total)


## 4. Late по паре

Соберите второй счётчик с теми же ключами.

In [ ]:
pair_late = {}  # TODO
assert set(pair_late) == set(pair_total)
assert sum(pair_late.values()) == int(df["is_late"].sum())


## 5. Доля late пары

In [ ]:
pair_rate = None  # TODO
assert set(pair_rate) == set(pair_total)
assert all(0 <= value <= 1 for value in pair_rate.values())


## 6. Минимальный размер сегмента

Оставьте пары с total >= 3 (или >=1, если данных мало).

In [ ]:
MIN_SIZE = 3 if len(df) >= 30 else 1
eligible = None  # TODO
assert all(pair_total[key] >= MIN_SIZE for key in eligible)
assert set(eligible) <= set(pair_total)


## 7. Выше глобальной доли

Список `(pair, rate, total)` по убыванию rate. Глобальная доля здесь — линия сравнения, а не доказательство причины задержек.

In [ ]:
global_rate = float(df["is_late"].mean())
hot_segments = None  # TODO
assert all(rate > global_rate and total >= MIN_SIZE for _, rate, total in hot_segments)
assert all(hot_segments[i][1] >= hot_segments[i + 1][1] for i in range(len(hot_segments) - 1))


## 8. Watchlist заказов

Из hot-сегментов соберите set ids, затем пересеките с late.

In [ ]:
watch = None  # TODO
assert isinstance(watch, set)
assert watch <= set(df.loc[df["is_late"].eq(1), "order_id"])


## 9. Самостоятельно: отчёт сегмента

In [ ]:
def segment_report(frame, seller_state, customer_state):
    # TODO: total, late, rate, order_ids
    ...


example_pair = next(iter(pair_total))
report = segment_report(df, *example_pair)
assert set(report) == {"total", "late", "rate", "order_ids"}
assert report["total"] == len(report["order_ids"])
assert 0 <= report["rate"] <= 1
